<a href="https://colab.research.google.com/github/bcarmoai/ML-and-AI/blob/main/RAG_Chapter3_Demo2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The purpose of this notebook is to test a basic AI model on medical texts. This notebook does not provide medical advice. Do not use any of the medical information shown in this notebook.

Install Dependencies

In [1]:
%pip install requests tqdm faiss-cpu transformers tensorflow sentence-transformers textblob gensim

Define and Write the Server Application

In [2]:
%%writefile app.py
from flask import Flask, request, jsonify
import faiss
import numpy as np
import requests
import zipfile
from pathlib import Path
from tqdm import tqdm
import re
from gensim.utils import simple_preprocess
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import torch
import gc
import json
import os
import logging

DATA_DIR = Path("./mimic_textbooks")
INDEX_FILE_PATH = "faiss_index.idx"
CHUNKED_DOCS_PATH = "chunked_documents.json"

def save_chunked_documents(documents, file_path=CHUNKED_DOCS_PATH):
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(documents, f)

def load_chunked_documents(file_path=CHUNKED_DOCS_PATH):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

def download_and_extract_zip(url, extract_to=DATA_DIR):
    extract_to.mkdir(parents=True, exist_ok=True)
    zip_path = extract_to / "textbooks.zip"
    response = requests.get(url, stream=True)
    with open(zip_path, "wb") as file:
        for chunk in tqdm(response.iter_content(chunk_size=1024), unit='KB'):
            if chunk:
                file.write(chunk)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_to)

def load_text_files(directory):
    texts = []
    for file_path in Path(directory).glob("*.txt"):
        with open(file_path, "r", encoding="utf-8") as file:
            texts.append(file.read())
    return texts

def clean_and_tokenize(text):
    text = re.sub(r'\s+', ' ', text)
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    tokens = simple_preprocess(text)
    return ' '.join(tokens)

def chunk_text(text, chunk_size=200):
    words = text.split()
    return [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

def get_embeddings_in_batch(texts, batch_size=16):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = retrieval_tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=512)
        device = next(retrieval_model.parameters()).device
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = retrieval_model(**inputs)
        batch_embeddings = torch.mean(outputs.last_hidden_state, dim=1).cpu().numpy()
        all_embeddings.extend(batch_embeddings)
    return np.array(all_embeddings)

def get_query_embedding(query):
    inputs = retrieval_tokenizer(query, return_tensors="pt", padding=True, truncation=True)
    device = next(retrieval_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = retrieval_model(**inputs)
    return torch.mean(outputs.last_hidden_state, dim=1).cpu().numpy()

def retrieve_documents(query, top_k=5):
    query_embedding = get_query_embedding(query).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    return [chunked_documents[idx] for idx in indices[0]]

def generate_response(query, context, max_new_tokens=500):
    input_text = f"User query: {query}\n\nContext:\n{context}\n\nAnswer:"
    inputs = generation_tokenizer(input_text, return_tensors="pt", padding=True, truncation=True)
    # Dynamically target the exact device where the model resides to prevent tensor mismatches
    device = generation_model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = generation_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return generation_tokenizer.decode(outputs[0], skip_special_tokens=True)

# Safe GPU clear using native PyTorch
def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

app = Flask(__name__)
logging.basicConfig(level=logging.DEBUG)

print("Clearing GPU memory...")
clear_gpu()

gpu_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Retrieval Setup
retrieval_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
retrieval_model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2").to(gpu_device)
logging.info("Retrieval model loaded in GPU")

if os.path.exists(CHUNKED_DOCS_PATH):
    logging.info("Loading chunked documents...")
    chunked_documents = load_chunked_documents()
else:
    logging.info("Downloading and processing dataset...")
    dataset_url = "https://www.dropbox.com/scl/fi/54p9kkx5n93bffyx08eba/textbooks.zip?rlkey=2y2c5x8y0uncnddichn9cmd7n&st=m290nmkk&dl=1"
    download_and_extract_zip(dataset_url)
    documents = load_text_files(DATA_DIR / "textbooks/en")
    cleaned_documents = [clean_and_tokenize(doc) for doc in documents]
    chunked_documents = []
    for doc in cleaned_documents:
        chunked_documents.extend(chunk_text(doc))
    save_chunked_documents(chunked_documents)

if os.path.exists(INDEX_FILE_PATH):
    logging.info("Loading FAISS index from disk...")
    index = faiss.read_index(INDEX_FILE_PATH)
else:
    logging.info("Generating FAISS index embeddings...")
    embeddings = get_embeddings_in_batch(chunked_documents, batch_size=128)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings.astype("float32"))
    faiss.write_index(index, INDEX_FILE_PATH)

# Park retrieval model on CPU to clear space for the heavy generative model
retrieval_model.to("cpu")
clear_gpu()
logging.info("Retrieval model moved to CPU, GPU cleared.")

# 2. Generative Setup
logging.info("Loading generation model...")
# trust_remote_code is disabled for security, and float16/auto-mapping prevents OOM CPU spillage
generation_tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct", trust_remote_code=False)
generation_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3.5-mini-instruct",
    trust_remote_code=False,
    torch_dtype=torch.float16,
    device_map="auto"
)
logging.info("Generation model successfully mapped and loaded.")
print("Initialization complete.")

@app.route("/chat", methods=["POST"])
def chat():
    try:
        data = request.get_json()
        user_query = data.get("query")
        max_tokens = data.get("max_tokens", 500)

        retrieved_docs = retrieve_documents(user_query)
        retrieved_text = " ".join(retrieved_docs)

        response_text = generate_response(user_query, retrieved_text, max_new_tokens=max_tokens)
        return jsonify({"response": response_text})
    except Exception as e:
        logging.error(f"Error occurred: {e}")
        return jsonify({"error": str(e)}), 500

if __name__ == "__main__":
    # debug=False prevents the Flask reloader from spawning duplicate background instances
    app.run(host="0.0.0.0", port=5000, debug=False)

Writing app.py


Map IP Address

In [19]:
import requests
from socket import gethostname, gethostbyname

hostname = gethostname()
ip = gethostbyname(hostname)

print(f"Hostname: {hostname}")
print(f"IP Address: {ip}")

Hostname: 897dc0b4510e
IP Address: 172.28.0.12


Safe Orchestration & Server Launch

<small>(This handles all port clearing, background spawning, and startup logging automatically)

In [4]:
import subprocess
import os
import signal
import time

print("=== Force-clearing all processes on port 5000 ===")
try:
    pid_data = subprocess.check_output(["lsof", "-t", "-i:5000"]).decode("utf-8").strip()
    if pid_data:
        for pid in pid_data.split("\n"):
            print(f"Killing zombie process {pid}...")
            os.kill(int(pid), signal.SIGKILL)
        print("Port 5000 cleared.")
except Exception:
    print("No active processes found on port 5000. Port is clear.")

time.sleep(2)
print("=== Starting background server app.py ===")
log_file = open('nohup.out', 'w')
process = subprocess.Popen(['python', 'app.py'], stdout=log_file, stderr=log_file)

print("Monitoring background startup logs (this may take up to 2-3 minutes for model loading)...")
start_time = time.time()
initialized = False

while time.time() - start_time < 300:
    if os.path.exists('nohup.out'):
        with open('nohup.out', 'r') as f:
            content = f.read()
            if "Initialization complete." in content or "* Running on" in content:
                initialized = True
                break
    time.sleep(5)

if initialized:
    print("\n[SUCCESS] Flask server successfully initialized and is listening!")
else:
    print("\n[ERROR] Server failed to initialize within 5 minutes. Last logs:")
    with open('nohup.out', 'r') as f:
        print(f.read()[-1000:])

=== Force-clearing all processes on port 5000 ===
No active processes found on port 5000. Port is clear.
=== Starting background server app.py ===
Monitoring background startup logs (this may take up to 2-3 minutes for model loading)...

[ERROR] Server failed to initialize within 5 minutes. Last logs:
ontent.com:443 "GET /cd/0/get/DLU0HjgneQ8Yef_XotGCw6AA8Sm1Q4uNhvztNE3fumMtpQU6v1obQChdVDUGBgA8R3C1T-3-ANSEWh8A7m1s-50HSRXNACpY97gfjzODJCD84lZwcXYRmC0-DgM9fvVs-sSMD-_miKcVKrmQwHUlisEF/file?dl=1 HTTP/1.1" 200 90235299

0KB [00:00, ?KB/s]
2945KB [00:00, 29391.10KB/s]
5885KB [00:00, 20062.85KB/s]
8193KB [00:00, 16910.37KB/s]
12289KB [00:00, 18601.55KB/s]
16385KB [00:00, 19104.74KB/s]
20481KB [00:01, 19638.79KB/s]
24577KB [00:01, 20005.96KB/s]
28673KB [00:01, 20160.48KB/s]
32769KB [00:01, 20324.08KB/s]
36865KB [00:01, 20428.50KB/s]
40961KB [00:02, 20468.27KB/s]
44450KB [00:02, 23083.10KB/s]
46937KB [00:02, 22084.84KB/s]
49250KB [00:02, 19451.59KB/s]
53249KB [00:02, 19679.96KB/s]
57345KB [00:02

In [8]:
import subprocess

def check_background_server(port=5000):
    try:
        # Find the process ID (PID) listening on the specified port
        pid_data = subprocess.check_output(["lsof", "-t", f"-i:{port}"]).decode("utf-8").strip()
        if pid_data:
            pids = pid_data.split("\n")
            print(f"The Flask process is running on port {port} with PID(s): {', '.join(pids)}.")
        else:
            print(f"No process is currently running on port {port}.")
    except subprocess.CalledProcessError:
        print(f"No active process detected on port {port}. The server is likely stopped.")
    except Exception as e:
        print(f"An error occurred while checking the server: {e}")

check_background_server()

The Flask process is running on port 5000 with PID(s): 10548.


In [6]:
import subprocess
import os

print("=== 1. Network Binding (Port 5000) ===")
try:
    lsof_data = subprocess.check_output(["lsof", "-i:5000"]).decode("utf-8").strip()
    print(lsof_data if lsof_data else "Port 5000 is open, but nothing is bound to it yet.")
except Exception:
    print("Port 5000 is currently clear.")

print("\n=== 2. Active Python Processes ===")
try:
    ps_data = subprocess.check_output(["ps", "aux"]).decode("utf-8")
    app_procs = [line for line in ps_data.split('\n') if 'python app.py' in line]
    if app_procs:
        print("Found active app.py process(es):")
        for p in app_procs: print(p)
    else:
        print("No app.py background process is currently running.")
except Exception as e:
    print(f"Error checking processes: {e}")

print("\n=== 3. Live Log Tail (Last 20 lines) ===")
if os.path.exists('nohup.out'):
    with open('nohup.out', 'r') as f:
        lines = f.readlines()
        if lines:
            for line in lines[-20:]:
                print(line.strip())
        else:
            print("Log file exists but is completely empty.")
else:
    print("Log file 'nohup.out' does not exist.")

=== 1. Network Binding (Port 5000) ===
Port 5000 is currently clear.

=== 2. Active Python Processes ===
No app.py background process is currently running.

=== 3. Live Log Tail (Last 20 lines) ===
DEBUG:httpcore.http11:receive_response_headers.started request=<Request [b'GET']>
DEBUG:httpcore.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Content-Type', b'application/json; charset=utf-8'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Date', b'Thu, 05 Nov 2026 17:30:44 GMT'), (b'content-encoding', b'gzip'), (b'ETag', b'W/"a86-vnlDJGskSnIitl9YYFIgeSLsBlg"'), (b'X-Powered-By', b'huggingface-moon'), (b'X-Request-Id', b'Root=1-6aecbd44-1c98eba7227c0c6e5434550c;41fece09-5cd9-4bee-95e0-1541a6b00124'), (b'RateLimit', b'"api";r=494;t=164'), (b'RateLimit-Policy', b'"fixed window";"api";q=500;w=300'), (b'cross-origin-opener-policy', b'same-origin'), (b'Referrer-Policy', b'strict-origin-when-cross-origin'), (b'Access-Control-Max-Age',

In [8]:
!dmesg -T | grep -i 'killed process'

In [9]:
!df -h /

Filesystem      Size  Used Avail Use% Mounted on
overlay         236G  8.0G  228G   4% /


In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("Attempting to download and load Phi-3.5-mini-instruct...")

try:
    # We test with the exact same parameters your server uses
    tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct", trust_remote_code=False)
    model = AutoModelForCausalLM.from_pretrained(
        "microsoft/Phi-3.5-mini-instruct",
        trust_remote_code=False,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    print("SUCCESS: The model downloaded and loaded into memory!")
except Exception as e:
    print(f"FAILED: {e}")

Attempting to download and load Phi-3.5-mini-instruct...


[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

SUCCESS: The model downloaded and loaded into memory!


In [2]:
%pip install bitsandbytes accelerate

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

print("Loading Phi-3.5 in 4-bit precision...")

# Configure 4-bit quantization with 16-bit compute for accuracy
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct", trust_remote_code=False)

# device_map="cuda" explicitly bans the model from spilling into CPU RAM
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3.5-mini-instruct",
    trust_remote_code=False,
    quantization_config=quantization_config,
    device_map="cuda"
)

print(f"SUCCESS: Model loaded strictly on GPU.")
print(f"Current VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Loading Phi-3.5 in 4-bit precision...


[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

SUCCESS: Model loaded strictly on GPU.
Current VRAM allocated: 2.11 GB


In [4]:
import gc
import torch

if 'model' in globals():
    del model
if 'tokenizer' in globals():
    del tokenizer

gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared for server launch.")

GPU memory cleared for server launch.


Define and Write the Server Application

<small>(Modified to use 4-bit quantization)

In [5]:
%%writefile app.py
from flask import Flask, request, jsonify
import faiss
import numpy as np
import requests
import zipfile
from pathlib import Path
from tqdm import tqdm
import re
from gensim.utils import simple_preprocess
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import gc
import json
import os
import logging

DATA_DIR = Path("./mimic_textbooks")
INDEX_FILE_PATH = "faiss_index.idx"
CHUNKED_DOCS_PATH = "chunked_documents.json"

def save_chunked_documents(documents, file_path=CHUNKED_DOCS_PATH):
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(documents, f)

def load_chunked_documents(file_path=CHUNKED_DOCS_PATH):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

def download_and_extract_zip(url, extract_to=DATA_DIR):
    extract_to.mkdir(parents=True, exist_ok=True)
    zip_path = extract_to / "textbooks.zip"
    response = requests.get(url, stream=True)
    with open(zip_path, "wb") as file:
        for chunk in tqdm(response.iter_content(chunk_size=1024), unit='KB'):
            if chunk:
                file.write(chunk)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_to)

def load_text_files(directory):
    texts = []
    for file_path in Path(directory).glob("*.txt"):
        with open(file_path, "r", encoding="utf-8") as file:
            texts.append(file.read())
    return texts

def clean_and_tokenize(text):
    text = re.sub(r'\s+', ' ', text)
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    tokens = simple_preprocess(text)
    return ' '.join(tokens)

def chunk_text(text, chunk_size=200):
    words = text.split()
    return [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

def get_embeddings_in_batch(texts, batch_size=16):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = retrieval_tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=512)
        device = next(retrieval_model.parameters()).device
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = retrieval_model(**inputs)
        batch_embeddings = torch.mean(outputs.last_hidden_state, dim=1).cpu().numpy()
        all_embeddings.extend(batch_embeddings)
    return np.array(all_embeddings)

def get_query_embedding(query):
    inputs = retrieval_tokenizer(query, return_tensors="pt", padding=True, truncation=True)
    device = next(retrieval_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = retrieval_model(**inputs)
    return torch.mean(outputs.last_hidden_state, dim=1).cpu().numpy()

def retrieve_documents(query, top_k=5):
    query_embedding = get_query_embedding(query).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    return [chunked_documents[idx] for idx in indices[0]]

def generate_response(query, context, max_new_tokens=500):
    input_text = f"User query: {query}\n\nContext:\n{context}\n\nAnswer:"
    inputs = generation_tokenizer(input_text, return_tensors="pt", padding=True, truncation=True)
    device = generation_model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = generation_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return generation_tokenizer.decode(outputs[0], skip_special_tokens=True)

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

app = Flask(__name__)
logging.basicConfig(level=logging.DEBUG)

print("Clearing GPU memory...")
clear_gpu()

gpu_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Retrieval Setup
retrieval_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
retrieval_model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2").to(gpu_device)
logging.info("Retrieval model loaded in GPU")

if os.path.exists(CHUNKED_DOCS_PATH):
    logging.info("Loading chunked documents...")
    chunked_documents = load_chunked_documents()
else:
    logging.info("Downloading and processing dataset...")
    dataset_url = "https://www.dropbox.com/scl/fi/54p9kkx5n93bffyx08eba/textbooks.zip?rlkey=2y2c5x8y0uncnddichn9cmd7n&st=m290nmkk&dl=1"
    download_and_extract_zip(dataset_url)
    documents = load_text_files(DATA_DIR / "textbooks/en")
    cleaned_documents = [clean_and_tokenize(doc) for doc in documents]
    chunked_documents = []
    for doc in cleaned_documents:
        chunked_documents.extend(chunk_text(doc))
    save_chunked_documents(chunked_documents)

if os.path.exists(INDEX_FILE_PATH):
    logging.info("Loading FAISS index from disk...")
    index = faiss.read_index(INDEX_FILE_PATH)
else:
    logging.info("Generating FAISS index embeddings...")
    embeddings = get_embeddings_in_batch(chunked_documents, batch_size=128)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings.astype("float32"))
    faiss.write_index(index, INDEX_FILE_PATH)

# Move retrieval model to CPU to save GPU memory
retrieval_model.to("cpu")
clear_gpu()
logging.info("Retrieval model moved to CPU, GPU cleared.")

# 2. Generative Setup (4-bit Quantized)
logging.info("Loading 4-bit generation model...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

generation_tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct", trust_remote_code=False)
generation_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3.5-mini-instruct",
    trust_remote_code=False,
    quantization_config=quantization_config,
    device_map="cuda"
)
logging.info("Generation model loaded strictly on GPU.")
print("Initialization complete.")

@app.route("/chat", methods=["POST"])
def chat():
    try:
        data = request.get_json()
        user_query = data.get("query")
        max_tokens = data.get("max_tokens", 500)

        retrieved_docs = retrieve_documents(user_query)
        retrieved_text = " ".join(retrieved_docs)

        response_text = generate_response(user_query, retrieved_text, max_new_tokens=max_tokens)
        return jsonify({"response": response_text})
    except Exception as e:
        logging.error(f"Error occurred: {e}")
        return jsonify({"error": str(e)}), 500

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)

Overwriting app.py


Safe Orchestration & Server Launch

<small>(This handles all port clearing, background spawning, and startup logging automatically)

In [6]:
import subprocess
import os
import signal
import time

print("=== Force-clearing all processes on port 5000 ===")
try:
    pid_data = subprocess.check_output(["lsof", "-t", "-i:5000"]).decode("utf-8").strip()
    if pid_data:
        for pid in pid_data.split("\n"):
            print(f"Killing zombie process {pid}...")
            os.kill(int(pid), signal.SIGKILL)
        print("Port 5000 cleared.")
except Exception:
    print("No active processes found on port 5000. Port is clear.")

time.sleep(2)
print("=== Starting background server app.py ===")
log_file = open('nohup.out', 'w')
process = subprocess.Popen(['python', 'app.py'], stdout=log_file, stderr=log_file)

print("Monitoring background startup logs (this may take up to 2-3 minutes for model loading)...")
start_time = time.time()
initialized = False

while time.time() - start_time < 300:
    if os.path.exists('nohup.out'):
        with open('nohup.out', 'r') as f:
            content = f.read()
            if "Initialization complete." in content or "* Running on" in content:
                initialized = True
                break
    time.sleep(5)

if initialized:
    print("\n[SUCCESS] Flask server successfully initialized and is listening!")
else:
    print("\n[ERROR] Server failed to initialize within 5 minutes. Last logs:")
    with open('nohup.out', 'r') as f:
        print(f.read()[-1000:])

=== Force-clearing all processes on port 5000 ===
No active processes found on port 5000. Port is clear.
=== Starting background server app.py ===
Monitoring background startup logs (this may take up to 2-3 minutes for model loading)...

[SUCCESS] Flask server successfully initialized and is listening!


Define the Client Endpoint

In [16]:
import requests

def ask_chatbot(query):
    response = requests.post(f"http://{ip}:5000/chat", json={"query": query, "max_tokens": 1500})
    return response.json().get("response", response.json())

Execute Queries

In [20]:
print(ask_chatbot("Which are the vessels that irrigate the leg?"))

User query: Which are the vessels that irrigate the leg?

Context:
its location the levator ani nerve is susceptible to injury through parturition and pelvic surgery such as during sacrospinous or iliococcygeus vaginal vault suspensions figure the ligaments and fascial support of the pelvic viscera the muscles of the urogenital diaphragm anteriorly reinforce the pelvic diaphragm and are intimately related to the vagina and the urethra they are enclosed between the inferior and superior fascia of the urogenital diaphragm the muscles include the deep transverse perineal and sphincter urethrae table blood vessels the pelvic blood vessels supply genital structures as well as the following urinary and tracts muscles of the abdominal wall pelvic oor and perineum buttocks and upper thighs fasciae other connective tissue and bones skin and other superficial structures classically vessels supplying organs are known as visceral vessels and those supplying supporting structures are called parieta

In [21]:
print(ask_chatbot("What do beta blockers do?"))

User query: What do beta blockers do?

Context:
they are an excellent source of firstline therapy especially for migraine sufferers the original formulation propranolol is highly lipid soluble and contributed to bothersome side effects such as depression sleep disturbances nightmares in the elderly and constipation in higher doses propranolol has relative lack of beta selectivity which promotes other undesirable phenomena formulations such as atenolol are water soluble are beta selective and have fewer side effects than propranolol at higher doses beta effects emerge there is no evidence to support speculation that beta selective agents may be safe for use in individuals who have asthma an advantage of watersoluble agents is longer halflife reduced dosing schedules improve compliance side effects of betablockers include an increase in triglyceride levels and decrease in highdensity lipoprotein hdl cholesterol and blunting of adrenergic release in response to hypoglycemia nsaids may dec

In [22]:
print(ask_chatbot("How do cells of the pancreas look like in the microscope?"))

User query: How do cells of the pancreas look like in the microscope?

Context:
scattered throughout the organ in cell groupings of varying size fig it is estimated that million to million islets constitute about to of the volume of the pancreas but are most numerous in the tail individual islets may contain only few cells or many hundreds of cells plate page their polygonal cells are arranged in short chapter digestive system iii liver gallbladder and pancreas pancr eas figure electron micrograph of the apical cytoplasm of several pancreatic acinar cells one pancreatic acinar cell is outlined by the dashed line nuclei of adjoining cells are evident at the bottom left and right of the electron micrograph the apical cytoplasm contains extensive rough endoplasmic reticulum rer mitochondria secretory granules and golgi profiles at the apices of these cells lumen is present into which the zymogen granules are discharged junctional complex jc is indicated near the lumen irregular cords that

In [23]:
print(ask_chatbot("How do I detect psychosis in a patient?"))

User query: How do I detect psychosis in a patient?

Context:
and requests that the physician or an entourage leave immediately rarely catatonic state will make its first appearance in these circumstances causal relationship between the psychosis and medical illness is sought but cannot be established the psychosis may have preceded the medical illness but was not recognized the diagnostic study of the psychiatric illness must then proceed along the lines suggested in chap close observation will usually disclose clear sensorium and relatively intact memory features that permit differentiation from an acute confusional or delirious state or dementia the syndromes themselves and their main clinical causes are the only satisfactory basis for classification until such time as their actual causes and pathophysiology are discovered see table the first step in diagnosis is to recognize that the patient is confused this is obvious in most cases but as pointed out earlier the mildest forms part